# Online Retail II — Exploratory Data Analysis

**Dataset:** Online Retail II (UK-based online retailer, transaction line items, Dec 2009–Dec 2011)
**Source:** https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci

This notebook follows the structure requested for the exercise: executive summary, data overview, data quality, exploratory analysis, findings, caveats, and next steps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Executive Summary

*(Fill this in last, after you've done the analysis below — it should be 4-6 sentences summarizing the most decision-relevant findings, not a recap of every step.)*

## 2. Data Overview

In [ ]:
# Update this path once you've downloaded the data.
# The Kaggle CSV version typically has a single file; the original UCI version
# is an .xlsx with two sheets ('Year 2009-2010', 'Year 2010-2011').
DATA_PATH = 'data/online_retail_II.csv'  # adjust as needed

if DATA_PATH.endswith('.csv'):
    df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1')
else:
    sheets = pd.read_excel(DATA_PATH, sheet_name=None)
    df = pd.concat(sheets.values(), ignore_index=True)

print(df.shape)
df.head()

In [ ]:
# Standardize column names (Kaggle version uses 'Customer ID' with a space; normalize to be safe)
df.columns = [c.strip() for c in df.columns]
rename_map = {'Customer ID': 'CustomerID'}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
df.dtypes

**What is a row?** Each row appears to be a single product line within an invoice — one `StockCode` at one `Quantity`/`Price` on one `InvoiceDate`, tied to an `Invoice` number and (usually) a `CustomerID`. Multiple rows share the same `Invoice` when a customer buys several distinct products in one order. That makes the grain **invoice line item**, not "order" and not "customer" — worth stating explicitly since aggregations later need to roll up correctly (e.g. revenue per *order* requires grouping by `Invoice` first).

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())
print('Unique invoices:', df['Invoice'].nunique())
print('Unique customers:', df['CustomerID'].nunique())
print('Unique stock codes:', df['StockCode'].nunique())
print('Unique countries:', df['Country'].nunique())

## 3. Data Quality Checks

In [ ]:
# Missing values
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)
missing.sort_values('missing_pct', ascending=False)

`CustomerID` and `Description` are the columns most likely to have gaps. A missing `CustomerID` means that transaction can't be attributed to a customer — worth deciding early whether those rows get excluded from customer-level analysis (they should) while still counting toward revenue totals (they still can).

In [ ]:
# Exact duplicate rows
dupe_count = df.duplicated().sum()
print(f'Exact duplicate rows: {dupe_count} ({dupe_count/len(df)*100:.2f}%)')
df[df.duplicated(keep=False)].sort_values(['Invoice', 'StockCode']).head(10)

In [ ]:
# Negative or zero quantities/prices — cancellations, returns, adjustments, or bad data
print('Rows with Quantity <= 0:', (df['Quantity'] <= 0).sum())
print('Rows with Price <= 0:', (df['Price'] <= 0).sum())

# Invoices starting with 'C' are documented as cancellations in this dataset
cancelled = df['Invoice'].astype(str).str.startswith('C')
print('Cancellation invoices (Invoice starts with C):', cancelled.sum())
df.loc[cancelled].head(5)

In [ ]:
# Non-product stock codes (postage, fees, adjustments, samples) mixed into the line items
non_numeric_codes = df.loc[~df['StockCode'].astype(str).str.match(r'^\d{5}')]['StockCode'].value_counts().head(15)
non_numeric_codes

In [ ]:
# Extreme outliers in quantity and price
print(df[['Quantity', 'Price']].describe())
print()
print('Top 10 largest quantities:')
print(df.nlargest(10, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'CustomerID']])

**Data quality summary (fill in after running the cells above):**
- Missing `CustomerID`: ~X% of rows — excluded from customer-level cuts, retained for revenue totals
- Duplicates: X exact duplicate rows found — [decision: drop / keep and note]
- Cancellations (`Invoice` starting with 'C'): X rows, representing returns/adjustments — excluded from "sales" analysis, but worth a separate cancellation-rate metric
- Non-product stock codes (POST, DOT, M, etc.): [list what you found] — excluded from product-level analysis
- Outliers: [note anything that looks like a data entry error vs. a legitimate bulk order]

## 4. Exploratory Analysis

In [ ]:
# Build a clean transactional view for revenue analysis: valid sales only
sales = df[(df['Quantity'] > 0) & (df['Price'] > 0) & (~cancelled)].copy()
sales['Revenue'] = sales['Quantity'] * sales['Price']
print(f'Clean sales rows: {len(sales):,} of {len(df):,} original rows')
sales['Revenue'].sum()

In [ ]:
# Revenue trend over time (monthly)
monthly = sales.set_index('InvoiceDate').resample('MS')['Revenue'].sum()
ax = monthly.plot(marker='o')
ax.set_title('Monthly Revenue')
ax.set_ylabel('Revenue (£)')
plt.tight_layout()
plt.show()

In [ ]:
# Top products by revenue
top_products = (sales.groupby('Description')['Revenue']
                 .sum()
                 .sort_values(ascending=False)
                 .head(15))
top_products

In [ ]:
# Revenue and order concentration by country
by_country = sales.groupby('Country').agg(
    revenue=('Revenue', 'sum'),
    orders=('Invoice', 'nunique'),
    customers=('CustomerID', 'nunique')
).sort_values('revenue', ascending=False)
by_country.head(15)

In [ ]:
fig, ax = plt.subplots()
by_country.head(10)['revenue'].sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Countries by Revenue')
ax.set_xlabel('Revenue (£)')
plt.tight_layout()
plt.show()

In [ ]:
# Customer-level view: order frequency and spend distribution (basic RFM building blocks)
customer_summary = sales.dropna(subset=['CustomerID']).groupby('CustomerID').agg(
    total_revenue=('Revenue', 'sum'),
    n_orders=('Invoice', 'nunique'),
    first_purchase=('InvoiceDate', 'min'),
    last_purchase=('InvoiceDate', 'max'),
    n_countries=('Country', 'nunique')
)
customer_summary['avg_order_value'] = customer_summary['total_revenue'] / customer_summary['n_orders']
customer_summary.describe()

In [ ]:
# How concentrated is revenue among top customers? (Pareto check)
sorted_rev = customer_summary['total_revenue'].sort_values(ascending=False)
cum_share = sorted_rev.cumsum() / sorted_rev.sum()
top_10pct_cutoff = int(len(sorted_rev) * 0.10)
print(f"Top 10% of customers ({top_10pct_cutoff}) generate {cum_share.iloc[top_10pct_cutoff]*100:.1f}% of revenue")

fig, ax = plt.subplots()
cum_share.reset_index(drop=True).plot(ax=ax)
ax.set_title('Cumulative Revenue Share by Customer Rank')
ax.set_xlabel('Customers, ranked by revenue (descending)')
ax.set_ylabel('Cumulative revenue share')
plt.tight_layout()
plt.show()

In [ ]:
# One-time vs. repeat customers
repeat_share = (customer_summary['n_orders'] > 1).mean()
print(f'Share of customers with more than one order: {repeat_share*100:.1f}%')

## 5. Findings and Hypotheses

*(Write 3–5 here after reviewing the charts/tables above. For each: state the finding, why it matters commercially, and your confidence level with a one-line reason — e.g. "high confidence, large sample and consistent across months" vs. "low confidence, driven by a handful of large orders from one customer.")*

1. **Finding:** ...
   **Why it matters:** ...
   **Confidence:** ...

2. **Finding:** ...

3. **Finding:** ...

## 6. Caveats

*(What could be misleading, uncertain, or incomplete? Consider: the ~X% of rows with no CustomerID and what that does to customer-level stats; whether cancellations were handled consistently; single-country dominance skewing "global" patterns; the dataset spanning a fixed historical window with no way to know if it's representative of current behavior; any manual judgment calls made in the data quality step above.)*

## 7. Recommended Next Steps

*(If this were a real client engagement — what would you actually do next? E.g. validate the CustomerID-missing rows with the source system, build a proper RFM/cohort analysis, segment by country and category, check for seasonality vs. one-off promotional spikes, etc.)*

## 8. Use of GenAI

*(Describe concretely: what you asked it to help with — e.g. scaffolding the notebook structure, suggesting data quality checks, drafting chart code — what you changed or rejected, and how you verified the outputs against the actual data rather than taking them on faith.)*